# 01c – Baselines (task 18)

Runs after the split (`01b_split.ipynb`) and before the models (`02_estimate.ipynb`), so the models can be compared with these results.
Saves `output/tables/task18_baselines.csv` and `output/tables/task18_oof_predictions.csv`.

## Baselines

Two simple approaches that every model must beat, scored on the same 5 folds and with the same measures as the models (task 20):
- **Training mean:** every building gets the average index of the training folds. All scores are equal, so the ranking is random; this is a sanity check (houses: about 38% above 450 in the top 20%, about 20% captured).
- **Expert rule:** every building gets the average index of its construction period × energy group in the training folds (feedback point 4: can the models beat it?). A combination not seen in training gets the overall mean.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from metrics import evaluate
from cv import summarize
from pipeline import ExpertRuleEncoder, RULE_COLUMNS

processed = Path("../data/processed")
tables = Path("../output/tables")
tables.mkdir(parents=True, exist_ok=True)

In [ ]:
data = pd.read_csv(processed / "building_clean.csv")
split = pd.read_csv(processed / "split.csv")
data = data.merge(split, on="EGID", how="inner", validate="one_to_one")

train = data[data["test"] == False].reset_index(drop=True)   # test set stays closed until task 26
train["fold"] = train["fold"].astype(int)

y = train["y"].to_numpy(dtype=float)
is_house = train["house"].to_numpy(dtype=bool)
folds = train["fold"].to_numpy()

In [ ]:
pred_mean = np.full(len(train), np.nan)   # one prediction per building, filled fold by fold
pred_rule = np.full(len(train), np.nan)
per_fold = {"Training mean": [], "Expert rule": []}

for k in np.unique(folds):            # every fold in split.csv, so a different number of folds also works
    tr, va = folds != k, folds == k   # training = all other folds, validation = fold k

    pred_mean[va] = y[tr].mean()      # training mean
    rule = ExpertRuleEncoder(smoothing=0).fit(train.loc[tr, RULE_COLUMNS], y[tr])   # table of averages from training only
    pred_rule[va] = rule.transform(train.loc[va, RULE_COLUMNS]).ravel()             # look up each validation building

    for name, pred in [("Training mean", pred_mean), ("Expert rule", pred_rule)]:
        result = evaluate(y[va], pred[va], is_house[va])
        result.insert(0, "fold", k)
        per_fold[name].append(result)

In [ ]:
rows = []
for name, parts in per_fold.items():
    summary = summarize(pd.concat(parts, ignore_index=True))   # mean and SE over the folds, as in task 20
    for _, s in summary.iterrows():
        rows.append({
            "model": name,
            "group": s["group"],
            "top 20%: above 450 (%)": round(s["share_above_top20_mean"] * 100, 1),
            "± SE": round(s["share_above_top20_se"] * 100, 1),
            "top 10%: above 450 (%)": round(s["share_above_top10_mean"] * 100, 1),
            "top 20%: captured (%)": round(s["captured_top20_mean"] * 100, 1),
            "MAE": round(s["mae_mean"], 1),
        })
baselines = pd.DataFrame(rows)
baselines

In [ ]:
baselines.to_csv(tables / "task18_baselines.csv", index=False)
oof = train[["EGID", "fold", "house", "y"]].copy()
oof["Training mean"] = pred_mean
oof["Expert rule"] = pred_rule
assert oof[["Training mean", "Expert rule"]].notna().all().all(), "a building has no prediction"
oof.to_csv(tables / "task18_oof_predictions.csv", index=False)
print("Saved task18_baselines.csv and task18_oof_predictions.csv:", len(oof), "buildings")
